# 📓 Notebook ML Modelling — Sompo Predict
## Sprint 2 · Semana 3 · Task 1 — Pré-processamento + Split Treino/Teste

**Autor:** Rafael (Gon) · Grupo T1 · FIAP × Sompo Seguros  
**Dataset:** Base SUSEP rural tratada pelo Guilherme (149 obs, 8 var, 0 nulos)  
**Alvo:** `CLASSIFICACAO_RISCO` (multiclasse: Baixo / Médio / Alto / Crítico)

---

### 🎯 Objetivo desta seção
Preparar os dados para treino: codificar categóricas, dividir em treino/teste (70/30) com estratificação, e deixar pronto para o baseline (Task 2) e para o notebook de Deep Learning (Task 3).

## 1. Imports e configuração

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Reprodutibilidade — sempre fixar seed para que os resultados sejam replicáveis
RANDOM_STATE = 42

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Setup OK")

## 2. Carregar a base limpa

> **Justificativa:** Base já tratada pelo Guilherme (limpeza de outliers via IQR, NA tratados, padronização). Fonte: SUSEP — estatísticas de seguros rurais. Atende exigência do Prof. Rodolfo (Statistical Computing) de **dados reais com fonte citada**.

In [ ]:
df = pd.read_csv("base_sompo_limpa.csv")
print(f"Shape: {df.shape}")
print(f"Nulos totais: {df.isnull().sum().sum()}")
df.head()

## 3. Distribuição da variável alvo

> **Justificativa:** Antes de splitar, é obrigatório olhar o balanceamento. Classes desbalanceadas exigem split **estratificado** para que treino e teste preservem a proporção original.

In [ ]:
print("Distribuicao absoluta:")
print(df["CLASSIFICACAO_RISCO"].value_counts())
print("\nDistribuicao percentual:")
print((df["CLASSIFICACAO_RISCO"].value_counts(normalize=True) * 100).round(1))

# Visualizacao
fig, ax = plt.subplots(figsize=(8, 4))
df["CLASSIFICACAO_RISCO"].value_counts().reindex(["Baixo","Medio","Alto","Critico"]).plot(
    kind="bar", color=["#22c55e","#eab308","#f97316","#dc2626"], ax=ax)
ax.set_title("Distribuicao da Classificacao de Risco - base SUSEP tratada")
ax.set_ylabel("Quantidade de equipamentos")
ax.set_xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Insight:** ~45% Baixo, ~28% Médio, ~19% Alto, ~8% Crítico. Desbalanceamento moderado — vamos estratificar no split e reportar **matriz de confusão** (não só accuracy) no baseline.

## 4. Separação de features (X) e alvo (y)

> **Justificativa:** `VALOR_INDENIZADO_BRL` é uma variável **posterior ao sinistro** — usá-la causaria *data leakage* (o modelo "vê o futuro"). Por isso, removida das features.

In [ ]:
# Variavel alvo
y_raw = df["CLASSIFICACAO_RISCO"]

# Features - removendo o alvo E a variavel de leakage
X = df.drop(columns=["CLASSIFICACAO_RISCO", "VALOR_INDENIZADO_BRL"])

print("Features mantidas:", list(X.columns))
print(f"Shape X: {X.shape} | Shape y: {y_raw.shape}")

## 5. Codificação do alvo (ordinal)

> **Justificativa:** Como o risco tem ordem natural (Baixo < Médio < Alto < Crítico), aplicamos label encoding **manual e ordenado** — não usamos `LabelEncoder` do sklearn porque ele ordena alfabeticamente, o que perderia o significado.

In [ ]:
mapa_risco = {"Baixo": 0, "Medio": 1, "Alto": 2, "Critico": 3}
# Lidar com acentos: criar versao normalizada
mapa_risco_acentos = {"Baixo": 0, "Medio": 1, "Medio ": 1, "Alto": 2, "Critico": 3,
                      "Médio": 1, "Crítico": 3}
y = y_raw.map(mapa_risco_acentos)

print("Mapeamento aplicado:")
for k, v in mapa_risco.items():
    print(f"  {k} -> {v}")
print(f"\nDistribuicao y codificado:\n{y.value_counts().sort_index()}")

## 6. Codificação das features categóricas

### 6.1 INTENSIDADE_SINISTRO (ordinal)
> **Justificativa:** Tem ordem natural (Leve < Moderado < Grave < Total) — label encoding manual.

In [ ]:
mapa_intensidade = {"Leve": 0, "Moderado": 1, "Grave": 2, "Total": 3}
X["INTENSIDADE_SINISTRO"] = X["INTENSIDADE_SINISTRO"].map(mapa_intensidade)
print(X["INTENSIDADE_SINISTRO"].value_counts().sort_index())

### 6.2 UF e RAMO_SUSEP (nominais)
> **Justificativa:** Não têm ordem natural — one-hot encoding evita criar relação artificial. `drop_first=True` para evitar **multicolinearidade** (dummy trap).

In [ ]:
X = pd.get_dummies(X, columns=["UF", "RAMO_SUSEP"], drop_first=True, dtype=int)

print(f"Shape final de X: {X.shape}")
print(f"\nColunas finais ({len(X.columns)}):")
for col in X.columns:
    print(f"  - {col}")
X.head()

## 7. Split treino/teste 70/30 estratificado

> **Justificativa do 70/30:** Com 149 obs, 70/30 dá ~104 treino e 45 teste — equilíbrio entre dados pra modelo aprender e amostra de teste estatisticamente válida.  
> **Justificativa do estratificado:** Preserva a proporção das 4 classes em ambos os conjuntos (crítico para a classe "Crítico" que tem só 12 obs).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y   # mantem proporcao das classes
)

print(f"Treino: {X_train.shape[0]} amostras")
print(f"Teste:  {X_test.shape[0]} amostras")
print(f"\nDistribuicao no treino:")
print(y_train.value_counts(normalize=True).sort_index().round(3))
print(f"\nDistribuicao no teste:")
print(y_test.value_counts(normalize=True).sort_index().round(3))
print("\nProporcoes preservadas - estratificacao OK")

## 8. Salvar artefatos para as próximas tasks

> **Justificativa:** Persistir os splits garante que o **baseline (Task 2)** e o **notebook de Deep Learning (Task 3)** usem **exatamente os mesmos dados** — pré-requisito para comparação justa entre modelos.

In [ ]:
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Artefatos salvos: X_train.csv, X_test.csv, y_train.csv, y_test.csv")
print("Prontos pra Task 2 (Decision Tree baseline) e Task 3 (MLP).")

---

## ✅ Checklist Task 1 — Pré-processamento (Card Trello)

- [x] Imputar mediana → **N/A** (base já chegou sem nulos do Guilherme, justificado)
- [x] One-hot encoding (UF, RAMO_SUSEP) com `drop_first=True`
- [x] Label encoding ordinal (INTENSIDADE_SINISTRO, CLASSIFICACAO_RISCO)
- [x] Remoção de variável com data leakage (VALOR_INDENIZADO_BRL)
- [x] Split 70/30 com estratificação
- [x] Cada escolha justificada em 1 frase ✔ (exigência do card)
- [x] Artefatos persistidos para Task 2 e Task 3

**Próximo passo:** Task 2 — Treinar Decision Tree (max_depth=5) + matriz de confusão